In [1]:
!pip install torch transformers==4.36.2 scikit-learn networkx nltk

In [2]:
import os
import zipfile
import numpy as np
import pandas as pd
import nltk
import networkx as nx
import torch
import torch.nn as nn

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from transformers import BertTokenizer, BertModel

nltk.download('stopwords')

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
zip_path = "rumor_detection_acl2017.zip"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/")

In [4]:
DATASET_NAME = "twitter15"
BASE_PATH = "/content/rumor_detection_acl2017"
DATA_PATH = f"{BASE_PATH}/{DATASET_NAME}"


In [5]:
labels = {}
with open(f"{DATA_PATH}/label.txt") as f:
    for line in f:
        label, tid = line.strip().split(":")
        if label == "true": labels[tid] = 1
        elif label == "false": labels[tid] = 0

texts = {}
with open(f"{DATA_PATH}/source_tweets.txt") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) < 2: continue
        tid, text = parts[0], parts[1]
        if tid in labels:
            texts[tid] = text


df = pd.DataFrame([(tid, texts[tid], labels[tid]) for tid in texts],
                  columns=["id", "text", "label"])

print("Dataset size:", len(df))

Dataset size: 742


In [6]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
MAX_LEN = 24

encoded = tokenizer(
    df['text'].tolist(),
    padding=True,
    truncation=True,
    max_length=MAX_LEN,
    return_tensors='pt'
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [12]:
G = nx.Graph()
tree_path = f"{DATA_PATH}/tree"


def extract_tweet_id(node):
    parts = node.replace("[", "").replace("]", "").replace("'", "").split(",")
    return parts[1].strip() if len(parts) >= 2 else None

for file in os.listdir(tree_path):
    with open(f"{tree_path}/{file}") as f:
        for line in f:
            try:
                parent, child = line.strip().split("->")
                p = extract_tweet_id(parent)
                c = extract_tweet_id(child)
                if p and c:
                    G.add_edge(p, c)
            except:
                continue

print("Graph nodes:", len(G.nodes()))

X_graph = []
for node in df['id']:
    if node in G:
        X_graph.append([G.degree(node), nx.clustering(G, node)])
    else:
        X_graph.append([0, 0])
X_graph = torch.tensor(X_graph, dtype=torch.float32)
y = torch.tensor(df['label'].values, dtype=torch.float32)

Graph nodes: 53642


In [18]:
class GETAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.lstm = nn.LSTM(768, 64, batch_first=True, bidirectional=True)
        self.graph_fc = nn.Linear(2, 16)
        self.fc = nn.Linear(128 + 16, 32)
        self.out = nn.Linear(32, 1)
        self.bert.requires_grad_(False)

    def forward(self, ids, mask, graph):
        x = self.bert(input_ids=ids, attention_mask=mask).last_hidden_state
        x, _ = self.lstm(x)
        text_feat = x[:, -1, :]
        graph_feat = torch.relu(self.graph_fc(graph))
        combined = torch.cat((text_feat, graph_feat), dim=1)
        x = torch.relu(self.fc(combined))
        return torch.sigmoid(self.out(x))


In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

skf = StratifiedKFold(n_splits=5)

accs, precs, recs, f1s = [], [], [], []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_graph, y)):
    print(f"\n===== Fold {fold+1} =====")

    model = GETAE().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
    loss_fn = nn.BCELoss()

    X_ids = encoded['input_ids']
    X_mask = encoded['attention_mask']
    BATCH_SIZE = 8
    labels = labels.float()

    train_idx_tensor = torch.tensor(train_idx)
    for epoch in range(2):
      model.train()

      perm = torch.randperm(len(train_idx_tensor))

      for i in range(0, len(train_idx_tensor), BATCH_SIZE):
        batch_indices = perm[i:i+BATCH_SIZE]
        batch_idx = train_idx_tensor[batch_indices]   # ✅ correct

        ids = X_ids[batch_idx].to(device)
        mask = X_mask[batch_idx].to(device)
        graph = X_graph[batch_idx].to(device)
        labels = y[batch_idx].to(device)
        labels = labels.unsqueeze(1)   # ✅ makes shape [batch_size, 1]

        outputs = model(ids, mask, graph)
        loss = loss_fn(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

      print(f"Epoch {epoch+1} done")





    model.eval()
    with torch.no_grad():
        preds = model(
            X_ids[test_idx].to(device),
            X_mask[test_idx].to(device),
            X_graph[test_idx].to(device)
        ).cpu().numpy()

    preds = (preds > 0.5).astype(int)
    y_true = y[test_idx].numpy()

    accs.append(accuracy_score(y_true, preds))
    precs.append(precision_score(y_true, preds))
    recs.append(recall_score(y_true, preds))
    f1s.append(f1_score(y_true, preds))



===== Fold 1 =====


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Epoch 1 done
Epoch 2 done


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



===== Fold 2 =====
Epoch 1 done
Epoch 2 done

===== Fold 3 =====


...

Epoch 1 done
Epoch 2 done


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



===== Fold 4 =====
Epoch 1 done
Epoch 2 done

===== Fold 5 =====


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Epoch 1 done
Epoch 2 done


In [21]:
print("\n===== FINAL RESULTS =====")
print("Accuracy:", np.mean(accs))
print("Precision:", np.mean(precs))
print("Recall:", np.mean(recs))
print("F1 Score:", np.mean(f1s))


===== FINAL RESULTS =====
Accuracy: 0.5188010157808816
Precision: 0.3592592592592593
Recall: 0.2960720720720721
F1 Score: 0.24914930031209098
